### Lab1 - Course Task: Tokenization Efficiency Benchmark

In this notebook, we analyze how different tokenization strategies affect language model performance. We trained three models using identical architectures and data, varying only the tokenizer to measure its direct impact. The experiment was run on a laptop with an Intel Core i7-13700H CPU (14 cores, 20 threads), 30 GB of RAM and an NVIDIA GeForce RTX 4070 Laptop GPU with 8 GB of VRAM, under Ubuntu Linux (Python 3.14, PyTorch 2.15 with CUDA).

In [1]:
import torch, os, platform
print(torch.__version__, torch.cuda.is_available())
print(platform.platform(), "|", os.cpu_count(), "CPU threads")
print(torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB VRAM")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

2.15.0+cu132 True
Linux-7.0.0-38-generic-x86_64-with-glibc2.43 | 20 CPU threads
NVIDIA GeForce RTX 4070 Laptop GPU | 7.6 GB VRAM


### Data Preparation

For this experiment, we use `plwiki` dataset from Speakleash.

The beginning of the file is ordered (the first articles start with the letter A), so we iterate through the entire file and retain each article with a probability of 5%.

The sampled texts undergo three processing steps:
- whitespace normalization: any sequence of spaces, tabs, and newlines is replaced with a single space, ensuring that all tokenizers process identical text,
- filtering: removal of duplicate texts and short documents (under 200 characters),
- document shuffling: the remaining documents are shuffled randomly.

The collected documents are split into training and evaluation sets. The training set is used to build the whitespace tokenizer vocabulary, to train the SentencePiece tokenizer and to train the models. The evaluation set is used strictly for metric evaluation. Documents are taken from the beginning of the shuffled list for the evaluation set until reaching a threshold of at least 1.5 million characters, the remainder forms the training set. The evaluation set slightly exceeds the threshold because entire documents are added.

In [2]:
import os
from speakleash import Speakleash

base_dir = os.getcwd()
sl = Speakleash(os.path.join(base_dir, "datasets"))

In [3]:
import random

DATASET = "plwiki"
KEEP_PROB = 0.05      #we're choosing the 5% of the documents randomly (around 90M chars)
NUM_CHARS = 1_500_000
SEED = 42

rng = random.Random(SEED)
docs = []
for doc in sl.get(DATASET).data:
    if rng.random() < KEEP_PROB:
        docs.append(" ".join(doc.split()))     #normalization, only spaces

docs = [d for d in dict.fromkeys(docs) if len(d) >= 200]   #no duplicates and very short ones
rng.shuffle(docs)

#adding docs to eval until NUM_CHARS is reached, the rest belongs to the train data
n, i = 0, 0
while n < NUM_CHARS:
    n += len(docs[i])
    i += 1
eval_docs, train_docs = docs[:i], docs[i:]

def info(name, docs):
    chars = sum(len(d) for d in docs)
    words = sum(len(d.split()) for d in docs)
    print(f"{name}: {len(docs)} documents, {chars:,} chars, {words:,} words")

info("train", train_docs)
info("eval", eval_docs)

100%|████████████████████████████████████████████████████████████████████████████████| 889M/889M [02:11<00:00, 6.78MiB/s]


train: 57639 documents, 89,773,656 chars, 12,418,896 words
eval: 833 documents, 1,500,118 chars, 207,561 words


### Used tokenizers

We configured three tokenizers to use the same vocabulary size:

In [4]:
from typing import Protocol

class Tokenizer(Protocol):
    vocab_size: int

    def encode(self, text: str) -> list[int]: ...
    def is_direct(self, word: str) -> bool: ...     # is word a one token (no <UNK>)
    def pieces(self, text: str) -> list[str]: ...   # tokens as strings

**Pre-trained (GPT-2):** A standard BPE tokenizer trained on general web content.

In [5]:
from transformers import GPT2TokenizerFast

class GPT2Tokenizer:
    def __init__(self):
        self.tok = GPT2TokenizerFast.from_pretrained("gpt2")
        self.vocab_size = len(self.tok)

    def encode(self, text):
        return self.tok.encode(text)

    def is_direct(self, word):
        return len(self.tok.encode(" " + word)) == 1   # GPT-2 encodes words with a leading space

    def pieces(self, text):
        return [self.tok.decode([i]) for i in self.encode(text)]   # "�" = fragment of a multi-byte character
gpt2_tok = GPT2Tokenizer()
N = gpt2_tok.vocab_size
print(N)

/home/janwo/Dokumenty/computational-linguistics/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


50257


**Customized Whitespace-based:** A custom tokenizer with a fixed vocabulary, using a special unknown token and trained on the top most frequent words from our training set.

In [6]:
import re, time
from collections import Counter

class WhitespaceTokenizer:
    #\w - sequence of letters (word) [^\w\s] - punctuation marks
    pattern = re.compile(r"\w+|[^\w\s]")

    def __init__(self, texts, vocab_size):
        counts = Counter(tok for t in texts for tok in self.split(t))
        words = [w for w, _ in counts.most_common(vocab_size - 1)]
        self.itos = ["<UNK>"] + words        #id to string, <UNK> id is 0
        self.stoi = {w: i for i, w in enumerate(self.itos)}  #string to id
        self.vocab_size = len(self.itos)

    def split(self, text):
        return self.pattern.findall(text)

    def encode(self, text):
        return [self.stoi.get(tok, 0) for tok in self.split(text)]     #return id=0 if <UNK>

    def is_direct(self, word):
        return word in self.stoi

    def pieces(self, text):
        return [self.itos[i] for i in self.encode(text)]

t0 = time.time()
ws_tok = WhitespaceTokenizer(train_docs, N)
ws_build_time = time.time() - t0
print(ws_tok.vocab_size, f"{ws_build_time:.1f} s")

50257 4.2 s


**SentencePiece:** A BPE subword tokenizer trained from scratch on sentences from our dataset.

In [7]:
import sentencepiece as spm

# train file: one sentence per line
with open("spm_train.txt", "w", encoding="utf-8") as f:
    for doc in train_docs:
        for sent in re.split(r"(?<=[.!?]) ", doc):
            f.write(sent + "\n")

t0 = time.time()
spm.SentencePieceTrainer.train(f"--input=spm_train.txt --model_prefix=spm_pl --vocab_size={N} --model_type=bpe --minloglevel=2")  # type: ignore
spm_train_time = time.time() - t0

I0000 00:00:1793706516.530453   12919 sentencepiece_trainer.cc:227] Running command: --input=spm_train.txt --model_prefix=spm_pl --vocab_size=50257 --model_type=bpe --minloglevel=2


In [8]:
class SentencePieceTokenizer:
    def __init__(self, model_file):
        self.sp = spm.SentencePieceProcessor(model_file=model_file)
        self.vocab_size = self.sp.get_piece_size()

    def encode(self, text):
        return self.sp.encode(text)

    def is_direct(self, word):
        ids = self.sp.encode(word)
        return len(ids) == 1 and ids[0] != self.sp.unk_id()

    def pieces(self, text):
        return self.sp.encode(text, out_type=str)

sp_tok = SentencePieceTokenizer("spm_pl.model")
print(sp_tok.vocab_size, f"{spm_train_time:.1f} s")

50257 9.7 s


In [9]:
tokenizers = {
    "GPT-2 BPE": gpt2_tok,
    "Whitespace": ws_tok,
    "SentencePiece": sp_tok,
}
assert all(t.vocab_size == N for t in tokenizers.values())

### Tokenization statistics

All statistics in this section are computed on the evaluation set, which was not used to build any of the tokenizers.

*Tokens/word* is the number of tokens divided by the number of whitespace-separated words. *Words encoded directly* are counted on sequences of letters and digits (`\w+`, punctuation excluded) a word is encoded directly if it is a single token and not `<UNK>`.
Throughput depends on the implementation (Rust, C++, Python) and not only on the algorithm.

In [10]:
import pandas as pd

n_words = sum(len(d.split()) for d in eval_docs)  #number of words in eval data

rows = []
for name, tok in tokenizers.items():
    t0 = time.time()
    n_tok = sum(len(tok.encode(d)) for d in eval_docs)
    dt = time.time() - t0
    rows.append({"tokenizer": name, "tokens": n_tok,
                 "tokens/word": n_tok / n_words, "tokens/s": n_tok / dt})

pd.DataFrame(rows).set_index("tokenizer")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1942 > 1024). Running this sequence through the model will result in indexing errors


,tokens,tokens/word,tokens/s
tokenizer,,,
GPT-2 BPE,737181,3.551635,1.391262e+06
Whitespace,255530,1.231108,4.023752e+06
SentencePiece,311857,1.502484,9.074835e+05


In [11]:
word_re = re.compile(r"\w+")
word_counts = Counter(w for d in eval_docs for w in word_re.findall(d))
n_occ = sum(word_counts.values())

rows = []
for name, tok in tokenizers.items():
    direct = {w for w in word_counts if tok.is_direct(w)}
    rows.append({"tokenizer": name,
                 "words direct [%]": 100 * sum(word_counts[w] for w in direct) / n_occ,
                 "distinct direct": len(direct),
                 "distinct total": len(word_counts)})

pd.DataFrame(rows).set_index("tokenizer")

,words direct [%],distinct direct,distinct total
tokenizer,,,
GPT-2 BPE,26.548698,3211,51590
Whitespace,85.219016,27562,51590
SentencePiece,78.979771,19627,51590


On the 1.5M-character evaluation set, SentencePiece efficiently encodes Polish text at 1.50 tokens/word, covering 79% of words as single tokens. GPT-2 performs worst at 3.55 tokens/word and direct coverage of just 26.5%, frequently splitting words into fragments or raw bytes due to its English bias. The whitespace tokenizer achieves 1.23 tokens/word and 85.2% direct coverage, but permanently loses all remaining words as <UNK>. Because each tokenizer breaks the exact same text into vastly different token counts, throughput in tokens per second cannot be directly compared.

### Language model

We train three identical language models, one per tokenizer. The model is a small decoder-only Transformer built from PyTorch's default `nn.TransformerEncoder` with a causal mask: 2 layers, 4 attention heads, `d_model=256`, feed-forward size 1024, dropout 0.1, context length of 128 tokens. Training uses AdamW (learning rate 3e-4), batch size 32 and bfloat16 autocast. All models start from the same random seed and are trained for the same number of epochs on the same documents, so the only difference between them is the tokenizer (the vocabulary size is `N` for every model).

There is no separate validation set. The number of epochs is fixed in advance and the evaluation set is used only for the final metrics. Documents are concatenated into one stream of tokens without separators.

In [12]:
import torch.nn as nn
import torch.nn.functional as F

SEQ, BATCH = 128, 32            # context length (in tokens) and batch size
D, HEADS, LAYERS, FF = 256, 4, 2, 1024
DROPOUT, LR = 0.1, 3e-4
EPOCHS = 3                      # same for all models

class LM(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.emb = nn.Embedding(vocab, D)
        self.pos = nn.Embedding(SEQ, D)
        layer = nn.TransformerEncoderLayer(D, HEADS, FF, DROPOUT, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, LAYERS, enable_nested_tensor=False)
        self.out = nn.Linear(D, vocab)

    def forward(self, x):
        T = x.size(1)
        h = self.emb(x) + self.pos(torch.arange(T, device=x.device))
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)   # causal mask
        return self.out(self.enc(h, mask=mask, is_causal=True))

In [13]:
def encode_docs(tok, docs):
    return torch.tensor([t for d in docs for t in tok.encode(d)], dtype=torch.long)

def batches(data, shuffle=True):
    n = (len(data) - 1) // SEQ                         # number of blocks of SEQ tokens
    starts = torch.arange(n, device=data.device) * SEQ
    if shuffle:
        starts = starts[torch.randperm(n, device=data.device)]
    for i in range(0, n, BATCH):
        idx = starts[i:i + BATCH, None] + torch.arange(SEQ, device=data.device)
        yield data[idx], data[idx + 1]                 # input and target shifted by one token

def train_model(model, data, epochs):
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    model.train()
    step, t0 = 0, time.time()
    for ep in range(epochs):
        for x, y in batches(data):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
            opt.zero_grad()
            loss.backward()
            opt.step()
            step += 1
            if step % 500 == 0:
                print(f"epoch {ep+1}, step {step}, loss {loss.item():.3f}, {time.time()-t0:.0f} s")
    return time.time() - t0

In [14]:
train_times = {}
for name, tok in tokenizers.items():
    data = encode_docs(tok, train_docs).to(device)
    print(f"{name}: {len(data):,} tokens in train")

    torch.manual_seed(SEED)                     # same initialisation for every model
    model = LM(N).to(device)
    train_times[name] = train_model(model, data, EPOCHS)
    torch.save(model.state_dict(), f"lm_{name.replace(' ', '_')}.pt")
    print(f"{name}: training time {train_times[name]:.0f} s\n")
    del model, data
    torch.cuda.empty_cache()

GPT-2 BPE: 43,895,884 tokens in train
epoch 1, step 500, loss 4.602, 25 s
epoch 1, step 1000, loss 4.215, 49 s
epoch 1, step 1500, loss 3.819, 73 s
epoch 1, step 2000, loss 3.710, 98 s
epoch 1, step 2500, loss 3.473, 122 s
epoch 1, step 3000, loss 3.557, 147 s
epoch 1, step 3500, loss 3.407, 171 s
epoch 1, step 4000, loss 3.218, 195 s
epoch 1, step 4500, loss 3.240, 220 s
epoch 1, step 5000, loss 3.261, 244 s
epoch 1, step 5500, loss 3.141, 268 s
epoch 1, step 6000, loss 3.377, 293 s
epoch 1, step 6500, loss 3.187, 317 s
epoch 1, step 7000, loss 3.218, 341 s
epoch 1, step 7500, loss 3.264, 366 s
epoch 1, step 8000, loss 3.191, 390 s
epoch 1, step 8500, loss 3.251, 414 s
epoch 1, step 9000, loss 3.125, 439 s
epoch 1, step 9500, loss 2.923, 463 s
epoch 1, step 10000, loss 3.121, 488 s
epoch 1, step 10500, loss 2.883, 512 s
epoch 2, step 11000, loss 2.919, 536 s
epoch 2, step 11500, loss 2.915, 561 s
epoch 2, step 12000, loss 2.988, 585 s
epoch 2, step 12500, loss 2.911, 609 s
epoch 2, st

The training time follows the number of tokens: 1567 s for GPT-2 (43.9M training tokens, about 32k steps), 664 s for SentencePiece (18.6M, about 13.6k steps) and 548 s for whitespace (15.3M, about 11.2k steps). The GPT-2 tokenization makes training 2.4-2.9 times more expensive for the same text. Tokenizer speed in tokens/s (4.0M whitespace, 1.39M GPT-2, 0.91M SentencePiece) is misleading, because the token counts differ. Per word it is 3.3M, 0.60M and 0.39M words/s, so SentencePiece is faster than GPT-2. Building the tokenizer took 4.2 s for whitespace and 9.7 s for SentencePiece (GPT-2 is pretrained).

### Perplexity

Token-level perplexity is not comparable across tokenizers, so the total negative log-likelihood of the evaluation text is normalised by the number of words and by the number of characters:

$$
\text{PPL}_\text{word} = \exp\left(\frac{L \cdot T}{W}\right), \qquad
\text{PPL}_\text{char} = \exp\left(\frac{L \cdot T}{C}\right)
$$

where $L$ is the mean negative log-likelihood per token (in nats) of the model on the evaluation set, $T$ is the number of tokens the tokenizer produces for that text, $W$ is the number of whitespace-separated words and $C$ is the number of characters. The evaluation text is split into non-overlapping blocks of 128 tokens, so the first tokens of every block have less context than the rest; the few tokens left after the last full block are not scored (their share is negligible).

In [19]:
import math

@torch.no_grad()
def mean_nll(model, data):
    # mean negative log-likelihood per token (nats) over non-overlapping blocks of SEQ tokens
    model.eval()
    total, count = 0.0, 0
    for x, y in batches(data, shuffle=False):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(x)
        total += F.cross_entropy(logits.flatten(0, 1).float(), y.flatten(), reduction="sum").item()
        count += y.numel()
    return total / count

eval_words = sum(len(d.split()) for d in eval_docs)
eval_chars = sum(len(d) for d in eval_docs)

rows = []
for name, tok in tokenizers.items():
    model = LM(N).to(device)
    model.load_state_dict(torch.load(f"lm_{name.replace(' ', '_')}.pt", map_location=device))
    data = encode_docs(tok, eval_docs).to(device)
    total_nll = mean_nll(model, data) * len(data)      # mean NLL per token x all tokens of the eval text
    rows.append({"tokenizer": name,
                 "word PPL": math.exp(total_nll / eval_words),
                 "char PPL": math.exp(total_nll / eval_chars)})
    del model, data
    torch.cuda.empty_cache()

ppl_df = pd.DataFrame(rows).set_index("tokenizer")
ppl_df["train time [s]"] = pd.Series(train_times)
ppl_df

,word PPL,char PPL,train time [s]
tokenizer,,,
GPT-2 BPE,12049.549342,3.669914,1566.673108
Whitespace,317.836165,2.219283,547.748024
SentencePiece,4839.432360,3.234742,663.800359


The whitespace model gets the best perplexity (word PPL 318, char PPL 2.22), followed by SentencePiece (4,839; 3.23) and GPT-2 (12,050; 3.67). However, the whitespace model shouldn't be directly compared to the others because it has an easier job, 14.8% of its words are just <UNK> tokens, so it never has to spell out unknown words.

The real comparison is between SentencePiece and GPT-2, which both process full text. SentencePiece clearly wins, because its word PPL is about 2.5x lower. This is mostly because SentencePiece creates shorter token sequences:. In a 128-token context, it sees ~85 words compared to GPT-2's ~36, whereas GPT-2 struggles to learn Polish spelling from tiny fragments using a small model.

### OOV statistics

Out-of-vocabulary words are counted for the whitespace tokenizer, on sequences of letters and digits in the evaluation set. Every OOV word is replaced by `<UNK>`, so the whitespace model never has to predict how such a word is spelled.

In [16]:
oov_types = {w for w in word_counts if not ws_tok.is_direct(w)}
oov_occ = sum(word_counts[w] for w in oov_types)
print(f"OOV words: {oov_occ:,} of {n_occ:,} ({oov_occ/n_occ:.2%})")
print(f"distinct OOV words: {len(oov_types):,} of {len(word_counts):,} ({len(oov_types)/len(word_counts):.2%})")

OOV words: 30,923 of 209,208 (14.78%)
distinct OOV words: 24,028 of 51,590 (46.57%)


30,923 of 209,208 word occurrences (14.8%) and 24,028 of 51,590 distinct words (46.6%) are out of the whitespace dictionary. Polish is highly inflected, so a 50k word list cannot hold all forms of all words (and the dictionary is case-sensitive). GPT-2 and SentencePiece can encode any word (BPE).

### Qualitative analysis

Three fragments of 40 words each, randomly chosen from the evaluation set. In the GPT-2 output `�` marks a fragment of a multi-byte character (Polish letters are split into bytes), in the SentencePiece output `▁` marks the beginning of a word.

In [17]:
import textwrap

long_docs = [d for d in eval_docs if len(d.split()) >= 60]
samples = [" ".join(d.split()[:40]) for d in random.Random(1).sample(long_docs, 3)]

for i, text in enumerate(samples, 1):
    print(f"--- sample {i} ---\n{textwrap.fill(text, 100)}\n")
    words = word_re.findall(text)
    for name, tok in tokenizers.items():
        pcs = tok.pieces(text)
        direct = sum(tok.is_direct(w) for w in words)
        print(f"{name}: {len(pcs)} tokens, {len(pcs)/len(text.split()):.2f} tokens/word, "
              f"{100*direct/len(words):.0f}% words direct")
        print(textwrap.fill("|".join(pcs), 100), "\n")

--- sample 1 ---
Przepona (łac. "diaphragma") – główny mięsień oddechowy występujący u ssaków, należy do mięśni
poprzecznie prążkowanych płaskich. Oddziela jamę brzuszną od jamy klatki piersiowej i stanowi jej
ścianę dolną. Przepona jest uwypuklona w stronę klatki piersiowej w formie dwóch kopuł -

GPT-2 BPE: 156 tokens, 3.90 tokens/word, 16% words direct
Pr|z|ep|ona| (|ł|ac|.| "|di|aph|rag|ma|")| –| g|ł|ó|wn|y| mi|�|�|sie|�|�| odd|ech|ow|y|
w|y|st|�|�|pu|j|�|�|cy| u| ss|ak|ó|w|,| n|ale|�|�|y| do| mi|�|�|�|�|ni| pop|r|z|ec|z|nie|
pr|�|�|�|�|k|ow|any|ch| p|ł|ask|ich|.| Odd|z|iel|a| jam|�|�| br|z|us|zn|�|�| od| j|amy| k|lat|ki|
p|iers|i|owe|j| i| st|an|ow|i| je|j| �|�|c|ian|�|�| d|oln|�|�|.| Pr|z|ep|ona| j|est|
u|w|yp|uk|l|ona| w| str|on|�|�| k|lat|ki| p|iers|i|owe|j| w| form|ie| dw|ó|ch| k|op|u|ł| - 

Whitespace: 48 tokens, 1.20 tokens/word, 79% words direct
<UNK>|(|łac|.|"|<UNK>|"|)|–|główny|mięsień|<UNK>|występujący|u|ssaków|,|należy|do|mięśni|poprzecznie
|<UNK>|płaskich|.|Oddziela|<U

GPT-2 splits Polish words into many small pieces. Letters such as ę, ą, ś are not in its vocabulary and are encoded as raw bytes (the `�` in the output), while others (ł, ó) become one-letter tokens. For example "mięsień" becomes `mi|�|�|sie|�|�`, and "główny" takes five tokens. Only 16-42% of words in the samples are encoded directly (2.6-3.9 tokens/word). Numbers are single tokens, and the English origin of the vocabulary shows (`history|k` for "historyk"). The whitespace tokenizer keeps whole words (1.1-1.25 tokens/word), but everything outside the dictionary becomes `<UNK>`: names (*Sanguinetto*, *Cancuén*, *Petén*), but also rarer or inflected Polish words (*oddechowy*, *brzuszną*, *Maltańskiego*). All unknown words share one token, so the model cannot tell that the `<UNK>` after "Ruiny" is the same city as the first word of the text. SentencePiece (1.4-1.65 tokens/word, 71-85% words direct) keeps frequent words whole and splits rare ones into pieces that often resemble real morphemes (`Namie|stnik`, `Malta|ńskiego`, `prąż|kowanych`). Nothing is lost, but names are cut arbitrarily (`Can|cu|én`) and so are numbers (`▁201|8`, `▁20|00.`).

### Summary

Whitespace is the fastest and gives the shortest sequences, but a fixed word list loses a lot for an inflected language. GPT-2 needs no training and has no OOV problem, but it is inefficient for Polish (long sequences, byte fragments). SentencePiece is the best compromise here: no real OOV, 2.4 times fewer tokens than GPT-2 and the best perplexity among the tokenizers that model the full text. The price is that it must be trained on a corpus of the target language.
